# trig unit circle

Lab 0.4 - Trigonometry & the Unit Circle, from scratch.
Build sine and cosine from ONE spinning point, prove they match NumPy, hit the classic
radians bug, then discover the same waves living inside a Transformer (positional encoding).

Run top-to-bottom:  python scripts/build.py --lab labs/trig_unit_circle/trig_unit_circle.py
Deterministic (np.random.seed). Plots saved to outputs/ as lab_0.4_*.png.

    pip install numpy matplotlib

In [ ]:
# notebook shim: kernels have no __file__; locate the repo root portably
import pathlib as _pl
try:
    __file__
except NameError:
    _here = _pl.Path.cwd().resolve()
    _root = next((_d for _d in [_here, *_here.parents]
                  if (_d / 'config' / 'channel.yaml').exists()), _here)
    __file__ = str(_root / 'labs' / 'trig_unit_circle.py')

import math
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
OUT = "outputs"
BLUE, CYAN, GREEN, RED, YEL = "#58a6ff", "#39d0d8", "#3fb950", "#f85149", "#f0b429"

# brand-matching DARK plot theme (so figures blend into the dark video, not white boxes)
plt.rcParams.update({
    "figure.facecolor": "#0b0e14", "axes.facecolor": "#0b0e14", "savefig.facecolor": "#0b0e14",
    "text.color": "#e6edf3", "axes.titlecolor": "#e6edf3", "axes.labelcolor": "#e6edf3",
    "xtick.color": "#9aa4b2", "ytick.color": "#9aa4b2", "axes.edgecolor": "#243042",
    "grid.color": "#1b2230", "legend.facecolor": "#0e1626", "legend.edgecolor": "#243042",
    "legend.labelcolor": "#e6edf3", "figure.dpi": 120,
})

## Section intro — The Unit Circle

In [ ]:
# A vector of length 1 sweeps around the origin (like a clock hand). The angle is in RADIANS.
theta = np.linspace(0, 2 * np.pi, 361)        # one full turn
px, py = np.cos(theta), np.sin(theta)          # the point's position as it walks
assert np.allclose(px**2 + py**2, 1.0)         # always distance 1 from the centre
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(px, py, color=CYAN)
ax.plot([0, np.cos(1.0)], [0, np.sin(1.0)], color=YEL, lw=3)   # the hand at t=1 rad
ax.scatter([np.cos(1.0)], [np.sin(1.0)], color=YEL, zorder=3)
ax.set_aspect("equal"); ax.set_title("The unit circle: a point at angle t")
plt.show()
print("1) unit circle: a point of radius 1, position (cos t, sin t)")

## Section shadows — Cosine and Sine as Shadows

In [ ]:
# Cosine and sine are NOT about triangles - they are the two shadows of the point.
t = 1.0
x_shadow, y_shadow = np.cos(t), np.sin(t)
print(f"2) at t={t} rad:  cos t (x-shadow) = {x_shadow:.3f},  sin t (y-shadow) = {y_shadow:.3f}")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(px, py, color="#243042")
ax.plot([0, x_shadow], [0, y_shadow], color=YEL, lw=2)
ax.plot([x_shadow, x_shadow], [0, y_shadow], "--", color=GREEN)    # drop to x-axis
ax.plot([0, x_shadow], [y_shadow, y_shadow], "--", color=RED)      # drop to y-axis
ax.scatter([x_shadow], [0], color=GREEN); ax.scatter([0], [y_shadow], color=RED)
ax.set_aspect("equal"); ax.set_title("cos = horizontal shadow, sin = vertical shadow")
plt.show()

## Section build — Build sin and cos From Scratch

In [ ]:
# How does a computer actually get these numbers? A series - add up shrinking terms.
def my_sin(t, terms=12):
    t = np.asarray(t, float); s = np.zeros_like(t)
    for n in range(terms):
        s += (-1) ** n * t ** (2 * n + 1) / math.factorial(2 * n + 1)
    return s

def my_cos(t, terms=12):
    t = np.asarray(t, float); s = np.zeros_like(t)
    for n in range(terms):
        s += (-1) ** n * t ** (2 * n) / math.factorial(2 * n)
    return s

grid = np.linspace(-np.pi, np.pi, 200)
assert np.allclose(my_sin(grid), np.sin(grid), atol=1e-6)   # our series == NumPy
assert np.allclose(my_cos(grid), np.cos(grid), atol=1e-6)
print("3) built sin & cos from a Taylor series - matches NumPy to 1e-6")

## Section identity — sin^2 + cos^2 = 1

In [ ]:
# Not magic - just Pythagoras on a circle whose radius never changes.
lhs = np.sin(grid) ** 2 + np.cos(grid) ** 2
assert np.allclose(lhs, 1.0)
print(f"4) sin^2 + cos^2 = {lhs.min():.6f} .. {lhs.max():.6f}  (always 1)")

## Section breakit — The Radians Trap

In [ ]:
# sin(90) is NOT 1 - because NumPy speaks RADIANS, not degrees.
wrong = np.sin(90)                 # 90 *radians* - nonsense
right = np.sin(np.radians(90))     # 90 degrees -> radians -> 1.0
print(f"5) np.sin(90) = {wrong:.3f}  (WRONG - that's 90 radians)")
print(f"   np.sin(radians(90)) = {right:.3f}  (correct)")
deg = np.linspace(0, 360, 361)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(deg, np.sin(deg), color=RED, label="sin(degrees)  # squished / wrong")
ax.plot(deg, np.sin(np.radians(deg)), color=GREEN, label="sin(radians(degrees))  # correct")
ax.legend(); ax.set_title("Feed degrees to np.sin and the wave breaks")
plt.show()

## Section knobs — Frequency and Phase

In [ ]:
# y = sin(2*pi*f*t + phase).  f = how fast it wiggles, phase = where it starts.
tt = np.linspace(0, 1, 500)
fig, ax = plt.subplots(figsize=(8, 3))
for f, ph, c in [(1, 0, CYAN), (3, 0, YEL), (1, np.pi / 2, GREEN)]:
    ax.plot(tt, np.sin(2 * np.pi * f * tt + ph), color=c, label=f"f={f}, phase={ph:.2f}")
ax.legend(); ax.set_title("Frequency squeezes; phase slides")
plt.show()
print("6) two knobs: frequency (cycles) and phase (shift)")

## Section positional — Hidden Inside AI

In [ ]:
# A language model sees words as an unordered bag. We tag each POSITION with a stack of
# sines & cosines at many frequencies - the "positional encoding" inside every Transformer.
def positional_encoding(n_pos=50, d=64):
    pos = np.arange(n_pos)[:, None]
    i = np.arange(d)[None, :]
    angle = pos / (10000 ** (2 * (i // 2) / d))
    pe = np.zeros((n_pos, d))
    pe[:, 0::2] = np.sin(angle[:, 0::2])     # even dims: sine
    pe[:, 1::2] = np.cos(angle[:, 1::2])     # odd dims:  cosine
    return pe

PE = positional_encoding()
assert PE.shape == (50, 64) and np.all(np.abs(PE) <= 1.0)
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(PE, aspect="auto", cmap="twilight")
ax.set_xlabel("encoding dimension (sin / cos at rising frequency)")
ax.set_ylabel("word position")
ax.set_title("Positional encoding = a wall of our sines & cosines")
fig.colorbar(im); plt.show()
print("7) positional encoding built from the very same sin & cos - this is inside GPT.")
print("\nDONE - from one spinning point to the trig hidden inside modern AI.")

## Section recap — In One Breath

In [ ]:
print("cos and sin are the shadows of a point on a circle;")
print("their squares always sum to one; they speak radians;")
print("frequency and phase shape them; and stacked together,")
print("they teach a machine the order of words.")